# Neural Minesweeper

A deep learning agent that learns to play Minesweeper from board states using PyTorch.

This project explores how neural networks can learn spatial reasoning patterns in Minesweeper by predicting mine probabilities and selecting optimal moves.

The goal is to treat Minesweeper as a machine learning problem rather than a purely logical puzzle.

## Playing by Mine Prediction

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple
import random

import numpy as np # will handle our board arrays
import matplotlib.pyplot as plt # for visualizations

import torch # used for neural network
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

Coord = Tuple[int, int] # a board cell is represented by [row, col]

#### Utility Functions
Small helper functions for the board navigation.

In [ ]:
def in_bounds(r: int, c: int, H: int, W: int) -> bool:
    '''checks if a position is valid on the board'''
    return 0 <= r < H and 0 <= c < W


def neighbors(cell: Coord, H: int, W: int) -> List[Coord]:
    '''returns all surrounding cells for a given square'''
    r, c = cell
    nbrs = []

    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue

            rr, cc = r + dr, c + dc
            if in_bounds(rr, cc, H, W):
                nbrs.append((rr, cc))

    return nbrs

**What this does**

- in_bounds(...) checks if a position is valid on the board.
- neighbors(...) returns all surrounding cells for a given square.

This is used everywhere:

- clue calculation
- logic inference
- counting surrounding mines

#### Enviornment result object
This stores the result of an opening cell

In [ ]:
@dataclass
class StepResult:
    hit_mine: bool
    clue: Optional[int]
    done: bool

**What this does**

Each move returns:

- hit_mine=True if the move lost the game
- clue=<number> if the cell was safe
- done=True if the game is over

#### Minesweeper environment
This is the game engine. It creates the board, places mines, computes clues, and handles moves.

In [ ]:
class MinesweeperEnv:
    """
    Minesweeper game environment.

    Key design choices:
    - Mines are placed after the first click so the first move is always safe.
    - If force_first_zero=True, mines are also blocked from the first-click neighborhood,
      making the first clue guaranteed to be 0.
    """

    def __init__(
        self,
        H: int = 22,
        W: int = 22,
        n_mines: int = 80,
        seed: Optional[int] = None,
        force_first_zero: bool = True,
    ):
        self.H = H
        self.W = W
        self.n_mines = n_mines
        self.force_first_zero = force_first_zero
        self.rng = random.Random(seed)

        self._mines = np.zeros((H, W), dtype=bool)
        self._revealed = np.zeros((H, W), dtype=bool)
        self._clues = np.zeros((H, W), dtype=np.int32)

        self._initialized = False
        self._done = False
        self._triggered_mines = 0
        self._opened_safe = 0

    def reset(self, n_mines: Optional[int] = None) -> None:
        if n_mines is not None:
            self.n_mines = n_mines

        self._mines.fill(False)
        self._revealed.fill(False)
        self._clues.fill(0)

        self._initialized = False
        self._done = False
        self._triggered_mines = 0
        self._opened_safe = 0

    @property
    def done(self) -> bool:
        return self._done

    @property
    def opened_safe(self) -> int:
        return self._opened_safe

    @property
    def triggered_mines(self) -> int:
        return self._triggered_mines

    @property
    def mines(self) -> np.ndarray:
        return self._mines

    @property
    def revealed(self) -> np.ndarray:
        return self._revealed

    @property
    def clues(self) -> np.ndarray:
        return self._clues

    def _compute_clues(self) -> None:
        for r in range(self.H):
            for c in range(self.W):
                if self._mines[r, c]:
                    self._clues[r, c] = -1
                else:
                    mine_count = 0
                    for rr, cc in neighbors((r, c), self.H, self.W):
                        if self._mines[rr, cc]:
                            mine_count += 1
                    self._clues[r, c] = mine_count

    def _place_mines(self, first: Coord) -> None:
        forbidden: Set[Coord] = {first}
        if self.force_first_zero:
            forbidden.update(neighbors(first, self.H, self.W))

        valid_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if (r, c) not in forbidden
        ]

        if self.n_mines > len(valid_cells):
            raise ValueError("Too many mines for the board configuration.")

        sampled_mines = self.rng.sample(valid_cells, self.n_mines)
        for r, c in sampled_mines:
            self._mines[r, c] = True

        self._compute_clues()
        self._initialized = True

    def open(self, cell: Coord) -> StepResult:
        if self._done:
            return StepResult(hit_mine=False, clue=None, done=True)

        r, c = cell
        if not in_bounds(r, c, self.H, self.W):
            raise ValueError(f"Out of bounds: {cell}")

        if self._revealed[r, c]:
            return StepResult(hit_mine=False, clue=int(self._clues[r, c]), done=False)

        if not self._initialized:
            self._place_mines(first=cell)

        self._revealed[r, c] = True

        if self._mines[r, c]:
            self._triggered_mines += 1
            self._done = True
            return StepResult(hit_mine=True, clue=None, done=True)

        clue = int(self._clues[r, c])
        self._opened_safe += 1

        if self._opened_safe == (self.H * self.W - self.n_mines):
            self._done = True

        return StepResult(hit_mine=False, clue=clue, done=self._done)

#### Logic bot
This bot plays using simple inference rules. It also acts as the “expert” that generates training data.

In [ ]:
class LogicBot:
    def __init__(self, env: MinesweeperEnv, seed: Optional[int] = None):
        self.env = env
        self.rng = random.Random(seed)

        self.cells_remaining: Set[Coord] = {
            (r, c) for r in range(env.H) for c in range(env.W)
        }
        self.inferred_safe: Set[Coord] = set()
        self.inferred_mine: Set[Coord] = set()
        self.clue_map: Dict[Coord, int] = {}

    def _unrevealed_neighbors(self, cell: Coord) -> List[Coord]:
        out = []
        for n in neighbors(cell, self.env.H, self.env.W):
            rr, cc = n
            if not self.env.revealed[rr, cc] and n not in self.inferred_mine:
                out.append(n)
        return out

    def _count_inferred_mines(self, cell: Coord) -> int:
        return sum((n in self.inferred_mine) for n in neighbors(cell, self.env.H, self.env.W))

    def _count_revealed_or_safe(self, cell: Coord) -> int:
        count = 0
        for n in neighbors(cell, self.env.H, self.env.W):
            rr, cc = n
            if self.env.revealed[rr, cc] or n in self.inferred_safe:
                count += 1
        return count

    def _infer_once(self) -> bool:
        changed = False

        for cell, clue in list(self.clue_map.items()):
            unrevealed = self._unrevealed_neighbors(cell)
            num_unrevealed = len(unrevealed)

            if num_unrevealed == 0:
                continue

            mines_inferred = self._count_inferred_mines(cell)

            # Rule 1: all hidden neighbors must be mines
            if (clue - mines_inferred) == num_unrevealed:
                for u in unrevealed:
                    if u not in self.inferred_mine:
                        self.inferred_mine.add(u)
                        self.cells_remaining.discard(u)
                        changed = True
                continue

            # Rule 2: all hidden neighbors must be safe
            total_neighbors = len(neighbors(cell, self.env.H, self.env.W))
            revealed_or_safe = self._count_revealed_or_safe(cell)

            if ((total_neighbors - clue) - revealed_or_safe) == num_unrevealed:
                for u in unrevealed:
                    if u not in self.inferred_safe:
                        self.inferred_safe.add(u)
                        self.cells_remaining.discard(u)
                        changed = True

        return changed

    def select_cell(self) -> Optional[Coord]:
        if self.inferred_safe:
            return self.inferred_safe.pop()

        if not self.cells_remaining:
            return None

        return self.rng.choice(tuple(self.cells_remaining))

    def step(self) -> Optional[Tuple[Coord, StepResult]]:
        cell = self.select_cell()
        if cell is None:
            return None

        self.cells_remaining.discard(cell)
        result = self.env.open(cell)

        if not result.hit_mine:
            assert result.clue is not None
            self.clue_map[cell] = result.clue
            self.inferred_safe.discard(cell)

            while self._infer_once():
                pass

        return cell, result

    def play(self, max_steps: int = 10_000, verbose: bool = False) -> Dict[str, int]:
        steps = 0

        while not self.env.done and steps < max_steps:
            move = self.step()
            if move is None:
                break

            cell, result = move
            steps += 1

            if verbose:
                if result.hit_mine:
                    print(f"step {steps}: opened {cell} -> mine")
                else:
                    print(f"step {steps}: opened {cell} -> clue={result.clue}")

            if result.hit_mine:
                break

        return {
            "opened_safe": self.env.opened_safe,
            "triggered_mines": self.env.triggered_mines,
            "steps": steps,
        }

#### Logic bot demo
A quick benchmark to make sure the rule-based generator behaves reasonably.

In [ ]:
def run_logic_bot_demo(n_games: int = 5, difficulty_mines: int = 80, seed: int = 0) -> None:
    rng = random.Random(seed)
    scores = []

    for _ in range(n_games):
        env = MinesweeperEnv(n_mines=difficulty_mines, seed=rng.randint(0, 10**9))
        bot = LogicBot(env, seed=rng.randint(0, 10**9))
        result = bot.play(verbose=False)
        scores.append(result["opened_safe"])

    avg_score = sum(scores) / len(scores)
    print(f"LogicBot demo | mines={difficulty_mines} | avg opened_safe={avg_score:.2f}")

**What this does**

It runs several games and reports average safe cells opened.

This is just a sanity check before generating data.

#### Board Encoding
This converts the current board into a 4-channel tensor that the neural network can understand.

In [ ]:
def board_to_tensor(env: MinesweeperEnv, bot: Optional[LogicBot] = None) -> np.ndarray:
    """
    Convert the current board state into a 4-channel tensor.

    Channels:
    0: normalized clue values for revealed cells
    1: revealed-cell mask
    2: inferred mine mask
    3: unknown-cell mask
    """
    H, W = env.H, env.W
    x = np.zeros((4, H, W), dtype=np.float32)

    inferred_mines = set()
    if bot is not None:
        inferred_mines = bot.inferred_mine

    for r in range(H):
        for c in range(W):
            if env.revealed[r, c]:
                x[0, r, c] = env.clues[r, c] / 8.0
                x[1, r, c] = 1.0
            elif (r, c) in inferred_mines:
                x[2, r, c] = 1.0
            else:
                x[3, r, c] = 1.0

    return x

**What this does**

The neural network cannot understand raw Minesweeper logic directly, so you give it structured channels:

- **Channel 0**: clue number
- **Channel 1**: whether a cell is revealed
- **Channel 2**: whether logic thinks it is a mine
- **Channel 3**: whether it is still unknown

This is the feature representation for the model.